# Sessione 5 · Grandi modelli di linguaggio

**Modulo 3 · Modelli e API** · 2 sessioni · LLM · prompt engineering · integrazione

> Entriamo nel territorio dei modelli di linguaggio (quelli dietro Claude, ChatGPT o Gemini). Oggi è una sessione soprattutto **concettuale e di pratica con i prompt**: capirai cosa sono, come funzionano internamente e come rivolgerti a loro bene. Il codice per chiamarli lo costruiremo nella Sessione 6.

**Al termine della sessione saprai:**
- Spiegare cos'è un Grande Modello di Linguaggio (LLM) e nominare le famiglie principali
- Capire, senza matematica, come funziona internamente
- Applicare tecniche di base di *prompt engineering* (l'arte di chiedere bene)
- Usare *few-shot*, *chain-of-thought* e *system prompt*

*Risultato della sessione: sai cos'è un LLM e scrivi prompt che funzionano.*

## 1. Cos'è un Grande Modello di Linguaggio (LLM)?

Un **Grande Modello di Linguaggio** (*Large Language Model*, LLM) è un modello di IA addestrato con quantità enormi di testo per svolgere un compito apparentemente semplice: **predire quale parola viene dopo**. Da questa capacità, portata all'estremo, emerge l'abilità di conversare, riassumere, tradurre o programmare.

Ricollegati a quanto già sai: un LLM è una **rete neurale profonda** (Sessione 2) gigantesca, costruita sull'architettura *Transformer* comparsa nel 2017 (Sessione 1). Il "grande" viene dalle sue dimensioni: miliardi di quei *pesi* che regola durante l'addestramento.

Le famiglie principali oggi sono:
- **Claude** — sviluppato da Anthropic
- **GPT** — sviluppato da OpenAI
- **Gemini** — sviluppato da Google
- **Llama** — sviluppato da Meta, con la particolarità di essere a pesi aperti (puoi scaricarlo ed eseguirlo tu)

> Dato utile per il corso: useremo **Claude** (Anthropic) come esempio principale e, in parallelo, **modelli gratuiti** (Llama tramite OpenRouter, e Llama in locale con Ollama). Così vedi che l'idea è la stessa indipendentemente dal fornitore.

## 2. Come funziona un LLM internamente (senza matematica)

Andiamo con calma, un pezzo alla volta.

**Passo 1 · Il testo viene spezzettato in *token*.** Un modello non vede "parole" come noi, ma pezzi chiamati **token**: frammenti di parola. Ad esempio, "incredibile" potrebbe dividersi in `incred` + `ibile`. Come regola molto approssimativa, in italiano **un token equivale più o meno a 3 o 4 lettere**. Questo conta perché il costo e i limiti delle API si misurano in token.

**Passo 2 · Predice il token successivo.** Dato un testo, il modello calcola quale token è più probabile che venga dopo, lo aggiunge, e ripete. Scrive **parola dopo parola**, come chi completa la frase "il cielo è di colore..." con "azzurro".

**Passo 3 · L'addestramento.** Prima di poter fare questo, il modello ha letto quantità immense di testo e ha regolato progressivamente i suoi pesi fino a predire bene. Dopodiché viene affinato per essere utile e seguire istruzioni.

**La conseguenza chiave da ricordare:** un LLM **non consulta un database di fatti**; genera la continuazione più probabile. Per questo a volte produce risposte che suonano sicure ma sono false. Questo fenomeno si chiama **allucinazione**, ed è il motivo per cui conviene verificare ciò che è importante.

### Cos'è la "finestra di contesto"?

Il modello "vede" solo una quantità limitata di testo alla volta: la sua **finestra di contesto**, misurata in token. Include la tua domanda più tutta la conversazione precedente che gli passi. Se esci da quella finestra, il modello "dimentica" ciò che è più vecchio.

Idea pratica: il modello **non ricorda** da solo le conversazioni precedenti. Se vuoi che abbia memoria di quanto detto, devi **rinviargli la cronologia** a ogni chiamata. Lo vedremo in codice nella Sessione 6.

## 3. Prompt engineering: l'arte di chiedere bene

Un **prompt** è l'istruzione che dai al modello. *Prompt engineering* è, semplicemente, **imparare a chiedere in modo da ottenere buoni risultati**. Non serve essere tecnici: è comunicazione chiara. Queste sono le tecniche di base che danno più risultati.

**Tecnica 1 · Sii specifico e dai contesto.** Più è chiaro il cosa, il per cosa e il per chi, meglio è.

| ❌ Prompt vago | ✅ Prompt specifico |
|---|---|
| "Scrivi sulla digitalizzazione" | "Scrivi un paragrafo di 5 righe, in tono divulgativo, che spieghi a una PMI del settore agroalimentare perché le conviene digitalizzare la sua tracciabilità" |

**Tecnica 2 · Indica il formato di output.** Chiedi esplicitamente la forma: "rispondi con un elenco di 3 punti", "restituisci solo un JSON con le chiavi nome e prezzo", "massimo 50 parole".

**Tecnica 3 · Suddividi i compiti complessi.** Se il compito è grande, dividilo in passi. Modelli e persone lavoriamo meglio così.

### Tecnica 4 · System prompt: assegnare un ruolo

Quasi tutte le API separano due tipi di messaggio:
- **System prompt** — istruzioni generali di comportamento: chi è il modello, il suo tono, le sue regole. Si fissa una volta e condiziona tutta la conversazione
- **User prompt** — la richiesta concreta di ogni turno

Esempio di *system prompt*:
> *"Sei un assistente esperto in trasformazione digitale per le PMI. Rispondi in italiano, con linguaggio semplice, evitando tecnicismi non necessari. Se non sei sicuro di un dato, lo dici."*

Con questo, tutte le risposte usciranno con quel ruolo e quel tono senza doverlo ripetere ogni volta.

### Tecnica 5 · Few-shot: insegnare con esempi

*Few-shot* significa dare **alcuni esempi già risolti** dentro il prompt affinché il modello imiti il pattern. È come mostrare due o tre casi prima di chiedergli il successivo.

Esempio per classificare il sentiment di un commento:

```
Classifica il sentiment come POSITIVO, NEGATIVO o NEUTRO.

Commento: "Il prodotto è arrivato in ritardo e mal imballato"  ->  NEGATIVO
Commento: "Funziona perfettamente, molto soddisfatto"          ->  POSITIVO
Commento: "È quello che mi aspettavo, niente di più"           ->  NEUTRO
Commento: "L'assistenza clienti è stata eccellente"             ->
```

Il modello, vedendo il pattern, completerà con `POSITIVO`. Quando dai **zero** esempi si chiama *zero-shot*; con **uno**, *one-shot*; con **più di uno**, *few-shot*.

### Tecnica 6 · Chain-of-thought: chiedere di ragionare passo dopo passo

Per problemi che richiedono ragionamento (calcoli, logica, decisioni con più fattori), chiedere al modello di **pensare passo dopo passo prima di rispondere** migliora molto i risultati. È la tecnica *chain-of-thought* (catena di pensiero).

Basta aggiungere qualcosa come:
> *"Ragiona passo dopo passo prima di dare la risposta finale."*

Invece di lanciare un numero che potrebbe essere sbagliato, il modello scompone il problema, e "pensando ad alta voce" indovina di più. È la stessa idea di quando a una persona chiedi di mostrare lo svolgimento invece del solo risultato.

### 🟢 Esercizio 1 · Riscrivi un prompt vago

Hai questo prompt debole:
> *"Parlami dell'IA in azienda"*

**Riscrivilo** applicando le tecniche 1 e 2: aggiungi contesto (pubblico, obiettivo) e specifica il formato di output. Provalo in un qualsiasi chatbot (Claude, ChatGPT...) e confronta la risposta con quella del prompt originale.

*Non c'è una soluzione unica: condividilo in classe e lo commentiamo.*

### 🟢 Esercizio 2 · Progetta un prompt few-shot

Vuoi classificare email come `URGENTE` o `NON URGENTE`. Scrivi un prompt few-shot con **3 esempi risolti** e una quarta email non risolta perché il modello la completi. Pensa quali esempi rappresentano bene ogni categoria.

## 4. Appendice opzionale · La tua prima chiamata (anticipo della Sessione 6)

Se vuoi iniziare a farti un'idea, questa cella mostra **come si presenterebbe** una chiamata a Claude da Python. **Non eseguirla ancora**: nella Sessione 6 installiamo tutto e configuriamo le chiavi con calma. Qui serve solo per toglierti la paura: nota come il *system prompt* e lo *user prompt* sono proprio quello che abbiamo appena visto.

```python
import anthropic

cliente = anthropic.Anthropic()   # usa la tua chiave dalla variabile d'ambiente

respuesta = cliente.messages.create(
    model="claude-haiku-4-5",
    max_tokens=300,
    system="Sei un esperto di trasformazione digitale. Rispondi in italiano e in modo chiaro.",
    messages=[{"role": "user", "content": "Spiega cos'è un LLM in 3 frasi."}],
)
print(respuesta.content[0].text)
```

Riconosci i pezzi: il ruolo va in `system`, la tua domanda in `messages`. **Tutto qui.** Il resto sono dettagli che padroneggeremo la prossima sessione.

## 5. Riepilogo della sessione

- Un **LLM** è una rete neurale enorme che impara a **predire il token successivo**; da qui emerge la sua capacità di linguaggio
- Non consulta fatti: **genera il più probabile**, per questo può **allucinare** ed è bene verificare
- Vede solo la sua **finestra di contesto**; non ricorda da solo (bisogna rinviargli la cronologia)
- *Prompt engineering* = chiedere bene: **sii specifico, indica il formato, assegna un ruolo** (*system prompt*), **fornisci esempi** (*few-shot*) e **chiedi di ragionare passo dopo passo** (*chain-of-thought*)

**Per la prossima sessione:** lo portiamo al codice. Chiamerai Claude e modelli gratuiti da Python, regolerai i parametri e costruirai il tuo primo chatbot.

👉 **Prima della Sessione 6, crea un account gratuito su Anthropic** (`console.anthropic.com`) **e un altro su OpenRouter** (`openrouter.ai`) per avere le tue chiavi pronte. Se il tuo computer è potente e vuoi l'opzione 100% locale, installa **Ollama** da `ollama.com`.